# DistilBERT Fine-Tuning on Google Colab

This notebook uses the project's corrected eight-category mapping. Copy the project folder, including `data/raw/tickets_bitext.csv` and `data/raw/extra_technical.csv`, to Google Drive at `/content/drive/MyDrive/ai-ticket-classifier`. In Colab, choose **Runtime > Change runtime type > T4 GPU** (or another GPU) first.

Categories: Account, Payment, Technical, Refund, Delivery, Subscription, General, Other. The training CSV includes 400 synthetic Technical starter messages; replace or supplement them with real, anonymized tickets before production. The audit below checks every CSV row against its source mapping.

In [ ]:
from pathlib import Path
import os
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/ai-ticket-classifier')
if not (PROJECT_DIR / 'ml' / 'train.py').is_file():
    raise FileNotFoundError(f'Project folder not found: {PROJECT_DIR}')
if not (PROJECT_DIR / 'data' / 'raw' / 'tickets_bitext.csv').is_file():
    raise FileNotFoundError('Copy data/raw/tickets_bitext.csv into the project folder on Drive first.')
os.chdir(PROJECT_DIR)
print('Project root:', Path.cwd())

In [ ]:
%pip install -q -r requirements.txt
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. Select a GPU runtime in Colab, then rerun this cell.')
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import json
import pandas as pd
from ml.prepare_bitext import ABCD_SUBFLOWS, CATEGORIES8, INTENT_TO_CATEGORY

csv_path = PROJECT_DIR / 'data' / 'raw' / 'tickets_bitext.csv'
report_path = PROJECT_DIR / 'data' / 'processed' / 'bitext_prep_report.json'
tickets = pd.read_csv(csv_path)
required = {'subject', 'message', 'category', 'source', 'intent', 'flags'}
assert required.issubset(tickets.columns), f'Missing columns: {required - set(tickets.columns)}'
assert set(tickets['category']) == set(CATEGORIES8), 'CSV does not contain exactly the eight configured categories'
assert tickets['message'].notna().all() and tickets['message'].str.strip().ne('').all(), 'Blank message found'

mismatches = []
for row in tickets.itertuples(index=False):
    if row.source == 'bitext':
        expected = INTENT_TO_CATEGORY.get(row.intent)
    elif row.source == 'abcd':
        flow, subflow = row.intent.removeprefix('abcd:').split('/', 1)
        mapping = ABCD_SUBFLOWS.get((flow, subflow))
        expected = mapping[0] if mapping else None
    elif row.source == 'extra':
        expected = 'Technical'
    else:
        expected = None
    if expected != row.category:
        mismatches.append((row.source, row.intent, row.category, expected))
assert not mismatches, f'Category mapping mismatches: {mismatches[:10]}'
extra_rows = tickets[tickets['source'].eq('extra')]
assert len(extra_rows) >= 300, f'Expected at least 300 extra Technical examples; found {len(extra_rows)}'
assert extra_rows['category'].eq('Technical').all()

report = json.loads(report_path.read_text(encoding='utf-8'))
counts = tickets['category'].value_counts().to_dict()
assert counts == report['rows_per_category'], 'CSV category counts differ from the preparation report'
print(f'PASS: {len(tickets):,} rows; all mappings match; {len(extra_rows)} extra Technical examples.')
print(pd.crosstab(tickets['category'], tickets['source'], margins=True))

In [ ]:
# Uses data/raw/tickets_bitext.csv and ml/labels_bitext8.json by default.
# The model is saved under models/ticket_classifier in the Drive project folder.
!python -m ml.train --require-gpu

In [ ]:
import json
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_dir = PROJECT_DIR / 'models' / 'ticket_classifier'
saved = json.loads((model_dir / 'labels.json').read_text(encoding='utf-8'))
labels = saved['labels']
tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
model = AutoModelForSequenceClassification.from_pretrained(str(model_dir)).to('cuda').eval()

# Replace or add these with your own anonymized tickets. These are spot checks, not evaluation data.
custom_tickets = [
    'I need to cancel order ORD-18907 before it ships.',
    'I am unhappy with the service and want to make a complaint.',
    'The mobile app crashes whenever I open the upload screen.',
    'My credit card is declined every time I try to pay.',
]
batch = tokenizer(custom_tickets, truncation=True, max_length=saved.get('max_length', 128), padding=True, return_tensors='pt').to('cuda')
with torch.inference_mode():
    probabilities = torch.softmax(model(**batch).logits, dim=-1)
for text, probs in zip(custom_tickets, probabilities):
    values, indices = torch.topk(probs, k=min(3, len(labels)))
    top = [(labels[index], round(value.item(), 4)) for value, index in zip(values, indices)]
    print(f'\nTicket: {text}\nPrediction: {top[0][0]}\nTop 3: {top}')

In [ ]:
# Evaluate only on the held-out test split created during training.
# Synthetic text and paraphrases across random splits can inflate metrics; review per-class recall.
# Technical has 1,118 rows, including 400 synthetic starters; Subscription has 1,417 rows.
!python -m ml.evaluate